# MobileNetV2 — Transfer Learning Classifier
### *Amphiprion ocellaris* detection | MLEES Master BEC CEE — University of Lausanne

Transfer learning approach using a pre-trained MobileNetV2 backbone (ImageNet weights) with a custom classification head, trained on the same GBIF dataset as the custom CNN.

**Pipeline:** data generators → frozen MobileNetV2 base → custom head → fine-tuning → evaluation → XAI

---

## 1. Imports

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    auc, classification_report, confusion_matrix,
    precision_score, recall_score, roc_curve,
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

sys.path.insert(0, "src")

## 2. Data generators
The dataset produced by `CNN.ipynb` (or `src/dataset.py`) is reused here. MobileNetV2 expects 224×224 inputs normalised to [0, 1].

In [ ]:
PROCESSED = Path("processed")
BATCH_SIZE = 32
IMG_SIZE   = (224, 224)

train_gen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
eval_gen = ImageDataGenerator(rescale=1.0 / 255.0)

kwargs = dict(target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="categorical")

train_generator = train_gen.flow_from_directory(PROCESSED / "train", **kwargs)
val_generator   = eval_gen.flow_from_directory(PROCESSED / "val",   **kwargs)
test_generator  = eval_gen.flow_from_directory(PROCESSED / "test",  shuffle=False, **kwargs)

num_classes  = len(train_generator.class_indices)
class_labels = list(train_generator.class_indices.keys())
print(f"Classes: {class_labels}")

## 3. Model architecture
MobileNetV2 base is frozen. A lightweight head (`GAP → Dropout → Dense → softmax`) is trained on top.

In [ ]:
Path("checkpoints").mkdir(exist_ok=True)

base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False

x       = GlobalAveragePooling2D()(base.output)
x       = Dropout(0.5)(x)
x       = Dense(128, activation="relu")(x)
outputs = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base.input, outputs=outputs)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
)

model.summary()

## 4. Training

In [ ]:
LOG_DIR = Path("logs/mobilenet")
LOG_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    ModelCheckpoint("checkpoints/mobilenet_best.keras", save_best_only=True, monitor="val_loss"),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.TensorBoard(log_dir=str(LOG_DIR)),
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=callbacks,
)

## 5. Evaluation
Loss / accuracy curves, confusion matrix, ROC-AUC, full classification report.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"],     label="train")
ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.legend()

ax2.plot(history.history["accuracy"],     label="train")
ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_preds  = np.argmax(model.predict(test_generator, verbose=0), axis=1)
test_labels = test_generator.classes

print(classification_report(test_labels, test_preds, target_names=class_labels))

cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
test_probs        = model.predict(test_generator, verbose=0)
test_labels_ohe   = tf.keras.utils.to_categorical(test_labels, num_classes=num_classes)

fpr, tpr, _ = roc_curve(test_labels_ohe.ravel(), test_probs.ravel())
roc_auc     = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Explainable AI — saliency maps
Shared `evaluate.py` utility — same visualisation as the custom CNN for a direct comparison.

In [ ]:
from evaluate import plot_saliency_grid

test_batch, _ = next(iter(test_generator))

plot_saliency_grid(
    model=model,
    images=test_batch,
    output_path=Path("results/mobilenet/saliency_grid.png"),
    n=4,
)

### Saliency on external images

In [ ]:
from PIL import Image
from evaluate import plot_saliency_grid

external_dir  = Path("data/external")
image_paths   = list(external_dir.glob("*.jpg")) + list(external_dir.glob("*.png"))

external_images = []
for p in image_paths:
    img = Image.open(p).resize((224, 224)).convert("RGB")
    external_images.append(np.array(img, dtype="float32") / 255.0)

if external_images:
    plot_saliency_grid(
        model=model,
        images=np.array(external_images),
        output_path=Path("results/mobilenet/saliency_external.png"),
        n=len(external_images),
    )
else:
    print("No images found in data/external/ — add .jpg or .png files there.")